# Free-threading lab — local CPU scaling

Original AGILAB notebook created September 19, 2026. BSD-3-Clause; see LICENSE and source/original.ipynb. [Python free-threading documentation](https://docs.python.org/3.14/howto/free-threading-python.html).

This measures the included unchanged AGILAB pool engine, not compatibility of the entire AGILAB dependency stack. All modes use the same free-threaded build, including the GIL-on control. No simulated or cached timings.

In [ ]:
# Stage 1: import and plan. PROJECT_ROOT and module search path are supplied.
from pathlib import Path
import json
import benchmark
from free_threading_core import tile_plan, reduce_tiles, image_digest
project_root = Path(PROJECT_ROOT)
assert (project_root / 'agilab_pool.py').is_file()
parameters = dict(width=96, height=64, iterations=100, workers=min(4, benchmark.effective_cpus()['effective_cpus']), repeats=1)
plan = tile_plan(parameters['width'], parameters['height'], parameters['iterations'])
print(f'{len(plan)} identical tiles; widths 1 and {parameters["workers"]}; three modes')


In [ ]:
# Stage 2: six real isolated cases, each with one repeat.
evidence = benchmark.run_benchmark(**parameters)
assert len(evidence['runs']) == 6
for row in evidence['summary']:
    print(row['label'], row['role'], row['workers'], 'workers:', round(row['wall_seconds'], 4), 's wall;', round(row['engine_seconds'], 4), 's engine')


In [ ]:
# Stage 3: independently recheck complete-image digests, then write a fresh artifact.
digests = [image_digest(reduce_tiles(run['records'], plan)) for run in evidence['runs']]
assert len(set(digests)) == 1 and digests[0] == evidence['digest']
assert evidence['same_work_verified']
results = {'results': {'local_cpu_scaling': evidence}}
Path('results.json').write_text(json.dumps(results, indent=2, allow_nan=False), encoding='utf-8')
print('Same-work verified across all six cases:', evidence['digest'])


Wall timing includes interpreter and pool startup, dispatch and reduction. Engine timing is reported separately. Speedups use each mode's one-worker median; with one repeat this is a single observation. Tiny workloads may be dominated by overhead, and one available CPU cannot establish scaling. Tile timestamps are recorded activity displayed after a run. Repeat with larger bounded workloads in the app; do not infer linear speedup or broad AGILAB compatibility.